In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

Found 292 branch digraphs...


Processing samples:  94%|█████████▍| 275/292 [00:07<00:00, 45.58it/s]/home/gaby/Lab/Libs/fundus-toolkits-common/src/fundus_toolkits/fundus_data.py:545: RuntimeWarning: The computed ROI mask is smaller than 60% of the image size and might be invalid.
  fundus_mask_ = fundus_ROI(fundus)  # Compute the fundus mask from the fundus image
Computing OD/Macula centers: 100%|██████████| 292/292 [00:26<00:00, 11.05it/s]


In [6]:
len(dataset)

292

## Graph Augment


In [ ]:
ID = 0
graph = dataset.graphs[ID]
fundus = FundusData(image=dataset.fundus_paths[ID])

In [ ]:
m, digraph, _ = dataset.draw_jppype(200, augment=True, test=True)
m

In [ ]:
digraph, fundus_img = dataset.get_sample(200, augment=True)

In [ ]:
import numpy as np

np.argmax([False, False, False, False, False])

In [ ]:
from fundus_toolkits.transform import ElasticTransform, IdentityTransform

digraph, fundus_img = dataset.get_sample(0)

%timeit dataset.get_sample(0, augment=True)

fundus_img = fundus_img.transpose(1, 2, 0)  # C,H,W -> H,W,C
shape = fundus_img.shape[0], fundus_img.shape[1]

elastic = ElasticTransform.random(shape, displacement_std=120, smoothing_size=200)
identity = IdentityTransform()
digraph.graph.transform(elastic, inplace=False)
%timeit digraph.graph.transform(elastic, inplace=False)
%timeit elastic.warp(fundus_img, warped_domain="same")

In [ ]:
len(digraph.graph.geometric_data().branch_curve())

In [ ]:
curve = digraph.graph.branch(0).curve()
elastic = ElasticTransform.random(shape, displacement_std=120, smoothing_size=200)
%timeit elastic.transform(curve)

In [ ]:
m.views[0].goto(dataset.get_digraph(0).graph.branch(24).midpoint().xy, 3)

In [ ]:
for i, data in enumerate(DataLoader(dataset, batch_size=8, num_workers=4)):
    ...